In [3]:
from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [4]:
!kaggle datasets download -d rmisra/news-category-dataset

Dataset URL: https://www.kaggle.com/datasets/rmisra/news-category-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 26.5M/26.5M [00:00<00:00, 148MB/s] 



In [5]:
import os

os.listdir('/content')

['.config', 'news-category-dataset.zip', 'sample_data']

In [6]:
!unzip -o /content/news-category-dataset.zip -d /content/dataset

Archive:  /content/news-category-dataset.zip
  inflating: /content/dataset/News_Category_Dataset_v3.json  


In [7]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


In [8]:
news_data = pd.read_json("/content/dataset/News_Category_Dataset_v3.json", lines=True)

#### Data Exploration

In [9]:
news_data.shape

(209527, 6)

In [10]:
news_data.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date'], dtype='object')

In [11]:
news_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [12]:
short_desc_data = news_data["short_description"].head(20000)

In [13]:
short_desc_data

,short_description
0,Health experts said it is too early to predict...
1,He was subdued by passengers and crew when he ...
2,"""Until you have a dog you don't understand wha..."
3,"""Accidentally put grown-up toothpaste on my to..."
4,Amy Cooper accused investment firm Franklin Te...
...,...
19995,He and his wife Grace would be granted full im...
19996,Transition is the name of the game in the Midd...
19997,The breakdown of coalition talks has plunged G...
19998,"In a major, thorough and truly important piece..."


#### Data preprocessing

In [14]:
short_desc_data[0]

'Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.'

In [15]:
short_desc_data.isnull().sum()

np.int64(0)

In [16]:
short_desc_data.duplicated().sum()

np.int64(66)

In [17]:
short_desc_data.drop_duplicates(inplace=True)

In [18]:
short_desc_data.duplicated().sum()

np.int64(0)

In [19]:
short_desc_data.shape

(19934,)

#### Tokenizing data

In [20]:
tokenizer = Tokenizer(oov_token="Nothing")

tokenizer.fit_on_texts(short_desc_data)

In [21]:
tokenizer.index_word

{1: 'Nothing',
 2: 'the',
 3: 'to',
 4: 'a',
 5: 'of',
 6: 'and',
 7: 'in',
 8: 'for',
 9: 'is',
 10: 'on',
 11: 'said',
 12: 'that',
 13: 'was',
 14: 'with',
 15: 'his',
 16: 'it',
 17: 'at',
 18: 'he',
 19: 'are',
 20: 'as',
 21: 'has',
 22: 'have',
 23: 'an',
 24: 'from',
 25: 'be',
 26: 'president',
 27: 'this',
 28: 'trump',
 29: '”',
 30: 'about',
 31: 'by',
 32: 'her',
 33: 'new',
 34: 'not',
 35: 'but',
 36: 'after',
 37: 'you',
 38: 'who',
 39: 'i',
 40: 'will',
 41: 'one',
 42: 'people',
 43: 'more',
 44: 'their',
 45: 'been',
 46: 'year',
 47: 'out',
 48: 'former',
 49: 'they',
 50: 'up',
 51: 'she',
 52: 'u',
 53: 'were',
 54: 's',
 55: 'all',
 56: 'we',
 57: 'just',
 58: 'what',
 59: 'first',
 60: 'over',
 61: 'had',
 62: 'house',
 63: 'than',
 64: 'says',
 65: 'when',
 66: 'two',
 67: 'no',
 68: 'would',
 69: 'or',
 70: 'some',
 71: 'like',
 72: 'state',
 73: "it's",
 74: 'into',
 75: 'show',
 76: 'if',
 77: 'also',
 78: 'time',
 79: 'say',
 80: 'last',
 81: 'your',
 82: 

In [22]:
tokenizer.word_counts

OrderedDict([('health', 203),
             ('experts', 106),
             ('said', 2294),
             ('it', 1482),
             ('is', 2832),
             ('too', 204),
             ('early', 71),
             ('to', 8110),
             ('predict', 6),
             ('whether', 107),
             ('demand', 27),
             ('would', 470),
             ('match', 16),
             ('up', 644),
             ('with', 1807),
             ('the', 21016),
             ('171', 1),
             ('million', 172),
             ('doses', 8),
             ('of', 6850),
             ('new', 1018),
             ('boosters', 1),
             ('u', 637),
             ('s', 622),
             ('ordered', 32),
             ('for', 3262),
             ('fall', 46),
             ('he', 1391),
             ('was', 2038),
             ('subdued', 1),
             ('by', 1042),
             ('passengers', 19),
             ('and', 5591),
             ('crew', 15),
             ('when', 492),
             (

In [23]:
len(tokenizer.word_index)

26343

In [24]:
data = tokenizer.texts_to_sequences(short_desc_data)

In [25]:
data[0]

[163,
 352,
 11,
 16,
 9,
 162,
 544,
 3,
 5026,
 347,
 1514,
 68,
 2396,
 50,
 14,
 2,
 13517,
 198,
 4125,
 5,
 2,
 33,
 13518,
 2,
 52,
 54,
 1298,
 8,
 2,
 897]

In [26]:
input_sequences = []

for seq in data:
    if len(seq) < 2:
        continue
    for i in range(1, len(seq)):
        ngram=seq[:i+1]
        input_sequences.append(ngram)

In [27]:
input_sequences

[[163, 352],
 [163, 352, 11],
 [163, 352, 11, 16],
 [163, 352, 11, 16, 9],
 [163, 352, 11, 16, 9, 162],
 [163, 352, 11, 16, 9, 162, 544],
 [163, 352, 11, 16, 9, 162, 544, 3],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026, 347],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026, 347, 1514],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026, 347, 1514, 68],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026, 347, 1514, 68, 2396],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026, 347, 1514, 68, 2396, 50],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026, 347, 1514, 68, 2396, 50, 14],
 [163, 352, 11, 16, 9, 162, 544, 3, 5026, 347, 1514, 68, 2396, 50, 14, 2],
 [163,
  352,
  11,
  16,
  9,
  162,
  544,
  3,
  5026,
  347,
  1514,
  68,
  2396,
  50,
  14,
  2,
  13517],
 [163,
  352,
  11,
  16,
  9,
  162,
  544,
  3,
  5026,
  347,
  1514,
  68,
  2396,
  50,
  14,
  2,
  13517,
  198],
 [163,
  352,
  11,
  16,
  9,
  162,
  544,
  3,
  5026,
  347,
  1514,
  68,
  2396,
  50,
  14,
  2,
  13

In [28]:
max_len = max([len (i) for i in input_sequences])

In [29]:
input_padding_sequence = pad_sequences(input_sequences, padding="pre", maxlen=max_len)

In [30]:
input_padding_sequence[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0, 163, 352], dtype=int32)

In [31]:
x=input_padding_sequence[:,:-1]
y=input_padding_sequence[:,-1]

In [32]:
y = np.array(y)

#### Modeling

In [33]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

In [34]:
vocab_size = len(tokenizer.word_index) + 1

In [35]:
vocab_size

26344

In [36]:
model = Sequential()

model.add(Embedding(vocab_size, 128, input_length=x.shape[1]))
model.add(LSTM(256))
model.add(Dense(vocab_size, activation="softmax"))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [37]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [38]:
model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"], optimizer="adam")

In [39]:
model.build(input_shape=(None, x.shape[1]))

In [40]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 43, 128)        │     3,372,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 26344)          │     6,770,408 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,536,680 (40.19 MB)

 Trainable params: 10,536,680 (40.19 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=50)

Epoch 1/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 130s 16ms/step - accuracy: 0.0874 - loss: 7.2814 - val_accuracy: 0.1125 - val_loss: 6.8963
Epoch 2/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 128s 17ms/step - accuracy: 0.1306 - loss: 6.3739 - val_accuracy: 0.1304 - val_loss: 6.7436
Epoch 3/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 121s 16ms/step - accuracy: 0.1582 - loss: 5.7911 - val_accuracy: 0.1386 - val_loss: 6.7689
Epoch 4/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 121s 16ms/step - accuracy: 0.1848 - loss: 5.2175 - val_accuracy: 0.1400 - val_loss: 6.9019
Epoch 5/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 120s 16ms/step - accuracy: 0.2206 - loss: 4.6483 - val_accuracy: 0.1372 - val_loss: 7.0881
Epoch 6/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 121s 16ms/step - accuracy: 0.2700 - loss: 4.1096 - val_accuracy: 0.1336 - val_loss: 7.2926
Epoch 7/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 120s 16ms/step - accuracy: 0.3255 - loss: 3.6307 - val_accuracy: 0.1283 - val_loss: 7.5079
Epoch 8/50
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 142s 16ms/step - accuracy: 

In [1]:
model.save("next_word_model.h5")

NameError: name 'model' is not defined

In [43]:
import pickle
with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)